# 4. Les LLM sans entraînement (*zero-shot*)

**Objectif** : classer les avis avec un **LLM**, **sans aucun entraînement**, puis le comparer à TF-IDF (93,8 %) et CamemBERT (95,8 %) sur trois critères : **précision, vitesse, coût**.

### Zero-shot, c'est quoi ?
| Approche | Ce qu'on fait | Exemples d'avis vus |
|---|---|---|
| TF-IDF + régression logistique | on entraîne un modèle de zéro | 160 000 |
| CamemBERT | on *fine-tune* un modèle pré-entraîné | 20 000 |
| **LLM zero-shot** | on **décrit la tâche en français** dans un message (le *prompt*), sans exemple | **0** |

Un LLM (*Large Language Model*) a été entraîné sur une énorme quantité de textes : il sait déjà ce qu'est une critique de film. On lui demande simplement « cet avis est-il positif ou négatif ? ». *Zero-shot* = « zéro exemple ». (Si on lui donnait quelques exemples corrigés dans le prompt, ce serait du *few-shot*.)

### Deux LLM très différents
| | **Qwen 2.5 (3B)** via Ollama | **Claude** via l'API d'Anthropic (optionnel, 4.7) |
|---|---|---|
| Où il tourne | sur **ton PC** | sur les serveurs d'Anthropic |
| Taille | 3 milliards de paramètres | beaucoup plus grand |
| Coût | gratuit | payant, au token |
| Confidentialité | rien ne sort de ton PC | les avis sont envoyés à l'API |

C'est un vrai choix d'ingénieur : on comparera les deux.

## 4.0 Vérifier Ollama

Ollama doit être lancé (tu l'as déjà utilisé pour le RAG). La cellule ci-dessous liste les modèles installés : `qwen2.5:3b` doit apparaître.

In [1]:
import json
import time

import numpy as np
import ollama
import pandas as pd

print([m.model for m in ollama.list().models])

['qwen2.5:3b']


## 4.1 Les données

On tire **500 avis au hasard** dans la validation (toujours les mêmes grâce à `random_state=42`). C'est assez pour mesurer, et le même échantillon servira pour Claude : la comparaison sera juste.

Avec 500 avis, l'accuracy sera moins précise qu'avec 20 000 : tu calculeras cette marge d'erreur au 4.4.

In [2]:
val = pd.read_parquet("data/validation.parquet")
echantillon = val.sample(500, random_state=42).reset_index(drop=True)

print(echantillon.shape)
print(echantillon["label"].value_counts())

(500, 2)
label
0    251
1    249
Name: count, dtype: int64


## 4.2 Le prompt et le premier appel

Un appel à un LLM de type « chat » contient :
- un **message système** : la consigne générale, le « rôle » ;
- un **message utilisateur** : ici, l'avis à classer.

Deux réglages importants :

| Réglage | Valeur | Pourquoi |
|---|---|---|
| `format` | un schéma JSON | **sorties structurées** : le modèle est *obligé* de répondre `{"sentiment": "positif"}` ou `{"sentiment": "negatif"}`. Sans ça, il pourrait répondre « Cet avis me semble plutôt positif… », difficile à exploiter automatiquement. |
| `temperature` | 0 | un LLM choisit chaque mot avec une part de hasard ; à 0, il prend toujours le plus probable : **mêmes entrées → mêmes réponses**, c'est reproductible. |

⏱️ **Le premier appel est lent (~30 s)** : Ollama charge le modèle en mémoire. Les suivants prennent moins d'une seconde.

La réponse contient aussi des **compteurs de tokens** : `prompt_eval_count` (tokens lus : consigne + avis) et `eval_count` (tokens écrits). Ici c'est gratuit, mais avec une API payante, c'est ce qu'on paie.

In [4]:
MODELE_LOCAL = "qwen2.5:3b"

CONSIGNE = (
    "Tu classes des critiques de films en français, publiées sur Allociné. "
    "Indique si l'avis global de l'auteur sur le film est positif ou négatif. "
    "Juge le verdict de l'auteur dans son ensemble, pas des phrases isolées."
)

SCHEMA = {
    "type": "object",
    "properties": {"sentiment": {"type": "string", "enum": ["positif", "negatif"]}},
    "required": ["sentiment"],
    "additionalProperties": False,
}


def appeler_ollama(texte):
    """Envoie un avis au modèle local et renvoie la réponse complète."""
    return ollama.chat(
        model=MODELE_LOCAL,
        messages=[
            {"role": "system", "content": CONSIGNE},
            {"role": "user", "content": texte},
        ],
        format=SCHEMA,
        options={"temperature": 0},
    )


avis = echantillon.loc[0, "review"]
print(avis[:300], "...")
print("Vraie réponse :", echantillon.loc[0, "label"])

reponse = appeler_ollama(avis)
print()
print("Texte renvoyé :", reponse.message.content)
print(reponse.prompt_eval_count, "tokens lus,", reponse.eval_count, "tokens écrits")

Un film extraordinaire par sa légèreté, sa poésie et cette vision calme et tranquille du monde. Tu vas au cinéma bien stressé après une journée de travail et tu en ressors apaisé et serein. Vraiment j'ai adoré! ...
Vraie réponse : 1

Texte renvoyé : { "sentiment" : "positif" }
131 tokens lus, 13 tokens écrits


### À TOI (4.2)
1. `reponse.message.content` est du **texte** qui ressemble à un dictionnaire. Transforme-le en vrai dictionnaire Python avec `json.loads(...)`, puis récupère la valeur de `"sentiment"`.
2. Convertis-la en **0 ou 1**, comme la colonne `label` (1 = positif).
3. Relance la cellule d'appel une 2ᵉ fois : combien de temps prend-elle maintenant ? Pourquoi le 1ᵉʳ appel était-il si lent ?

In [33]:
# À TOI
s=json.loads(reponse.message.content)
prediction=int(s["sentiment"]=="positif")

print(prediction)

1


**Mes réponses (4.2)** : 0 sec

## 4.3 La fonction `classer` et un essai sur 20 avis

Avant de lancer une longue boucle, on fait toujours un **essai pilote** sur quelques exemples : ça vérifie que tout marche et ça permet d'estimer la durée totale. (Avec une API payante, ça permet aussi d'estimer le coût **avant** de dépenser : tu le feras au 4.7.)

### À TOI (4.3)
1. Écris une fonction `classer(texte)` qui renvoie un **dictionnaire** :
   ```python
   {"prediction": 1 ou 0, "secondes": ..., "tokens_entree": ..., "tokens_sortie": ...}
   ```
   Pour mesurer le temps : `debut = time.perf_counter()` avant l'appel, puis `time.perf_counter() - debut` après.
2. Applique-la aux **20 premiers avis** de `echantillon` avec une boucle, et range les résultats dans un DataFrame : `pd.DataFrame(liste_de_dictionnaires)`.
3. Affiche l'accuracy sur ces 20 avis et la latence moyenne.
4. **Estime** la durée pour 500 avis, puis pour 20 000.

*Rappel* : une proportion se calcule comme au 3.3, avec `(... == ...).mean()`.

In [43]:
# À TOI
from sklearn.metrics import accuracy_score
def classer(texte):
    debut = time.perf_counter()
    reponse=appeler_ollama(texte)
    secondes=time.perf_counter() - debut
    s=json.loads(reponse.message.content)
    prediction=int(s["sentiment"]=="positif")
    return {"prediction": prediction, 
            "secondes": secondes, 
            "tokens_entree": reponse.prompt_eval_count, 
            "tokens_sortie": reponse.eval_count}

liste_de_dictionnaires=[classer(echantillon.loc[i, "review"]) for i in range(20)]
resultats = pd.DataFrame(liste_de_dictionnaires)
vrais = echantillon["label"][:20]      # les vraies réponses des 20 mêmes avis   
accuracy=accuracy_score(vrais, resultats["prediction"])
print(resultats["secondes"].mean())

0.24713025000019115


**Mes estimations (4.3)** : latence moyenne ≈ 0,25 s par avis, donc :
- 500 avis → 500 × 0,25 ≈ 125 s ≈ **2 minutes** ;
- 20 000 avis → 20 000 × 0,25 ≈ 4 900 s ≈ **1 h 20**.

## 4.4 L'évaluation sur 500 avis

### À TOI (4.4)
1. Même boucle qu'au 4.3, sur tout `echantillon` (≈ 5 minutes). Pour voir l'avancement, remplace `range(...)` par `tqdm(range(...))` (`from tqdm.auto import tqdm`).
2. **Sauvegarde** les résultats : `resultats.to_csv("data/qwen_500.csv", index=False)`. Si le noyau redémarre, tu relis le fichier avec `pd.read_csv(...)` au lieu de tout relancer.
3. Calcule l'accuracy et la latence moyenne.
4. **La marge d'erreur** : avec seulement 500 avis, l'accuracy mesurée est approximative. Un intervalle de confiance à 95 % vaut environ $p \pm 1{,}96\sqrt{\dfrac{p(1-p)}{n}}$, avec $p$ l'accuracy et $n = 500$. Calcule-le. Qwen est-il **vraiment** moins bon (ou meilleur) que CamemBERT (95,8 %) ?

In [45]:
# À TOI

liste_de_dictionnaires_500=[classer(echantillon.loc[i, "review"]) for i in range(len(echantillon))]
resultats = pd.DataFrame(liste_de_dictionnaires_500)
#resultats.to_csv("data/qwen_500.csv", index=False)
vrais = echantillon["label"]      
accuracy=accuracy_score(vrais, resultats["prediction"])
print(resultats["secondes"].mean())
print(accuracy)
marge=1.96 * np.sqrt(accuracy * (1 - accuracy) / 500)
print(marge)

0.2657458730001526
0.954
0.018362177125820354


**Ma conclusion (4.4)** : Qwen 2.5 3B obtient **95,4 %** sur 500 avis (intervalle de confiance à 95 % : 93,6 % – 97,2 %), avec une latence d'environ 0,27 s par avis sur mon PC. CamemBERT (95,8 %) est dans cet intervalle : **sur 500 avis, on ne peut pas dire lequel est meilleur**. En revanche, Qwen dépasse TF-IDF bigrammes (93,8 %, sous la borne basse), alors qu'il n'a vu **aucun** avis étiqueté.

Remarque : deux exécutions ont donné 95,6 % puis 95,4 %, malgré la température 0 (les calculs parallèles du processeur ne sont pas parfaitement reproductibles) ; d'où l'intérêt de sauvegarder les résultats et de les relire au lieu de relancer.

## 4.5 Les erreurs du modèle

Les chiffres ne disent pas tout. Regarde **les avis où Qwen s'est trompé**.

### À TOI (4.5)
1. Ajoute les colonnes de `resultats` à `echantillon` (`pd.concat([echantillon, resultats], axis=1)`), puis filtre les lignes où `prediction != label`.
2. Ses erreurs sont-elles équilibrées ? Compte combien sont des **faux positifs** (avis négatif classé positif) et combien des **faux négatifs**.
3. Lis au moins 5 de ces avis en entier. Pour chacun : est-ce **le modèle qui se trompe**, ou l'**étiquette qui est discutable** (avis mitigé, note qui ne correspond pas au texte) ?

Ce travail d'analyse est très apprécié en entretien : il montre que tu ne fais pas confiance aveuglément à une métrique.

In [60]:
complet = pd.concat([echantillon, resultats], axis=1)
masque = complet["prediction"] != complet["label"]   # True/False pour chaque ligne
erreurs = complet[masque]                            # on garde seulement les lignes True
print(erreurs["label"].value_counts())
for texte in erreurs["review"].head(5):
    print(texte)
    print("-" * 80)

label
0    13
1    10
Name: count, dtype: int64
Un film dynamique, un bon scénario qui donne envie de voir la suite, de bons acteurs. Tout à fait sympathique pour des adultes. A mon avis,le film est adapté à partir de 14/16 ans: langage très grossier et situations inadaptées aux plus jeunes, violence... Nous nous attendions à un film un peu plus soft pour un "tout public", la bande annonce n'était pas dans ce ton.
--------------------------------------------------------------------------------
Un film absurde, un scenario incohérent, des acteurs digne des meilleurs nanars, un bon milliard de faux raccords .... Et une des meilleurs comédies des années 80
--------------------------------------------------------------------------------
un petit film sympatique un peux avec le principe de saw en beaucoup moin bien
--------------------------------------------------------------------------------
Sans être un grand film,Belphégor est un film agréable, sans prétention et qui n'est jamais ridic

**Mon analyse (4.5)** : Qwen fait 23 erreurs sur 500, à peu près équilibrées : **13 faux positifs** (avis négatifs classés positifs) et **10 faux négatifs** (avis positifs classés négatifs). Il n'a donc pas de biais net vers une classe.

J'ai lu 5 erreurs en entier :

| # | Étiquette | Qwen | Avis (résumé) | Qui se trompe ? |
|---|---|---|---|---|
| 33 | positif | négatif | « bon scénario, bons acteurs, sympathique », puis une remarque sur l'âge conseillé (langage grossier) | **Qwen** : distrait par une critique secondaire qui ne porte pas sur la qualité du film |
| 41 | positif | négatif | une liste de défauts (« absurde, incohérent, nanar ») puis « une des meilleures comédies des années 80 » | **Qwen** : le verdict, affectueux, arrive à la fin, après les défauts |
| 140 | négatif | positif | « tro bi1 ce Film, C un cher Deuvre comme Vou Dite… lol » | **Qwen**, mais c'est difficile : **ironie** en langage SMS, prise au premier degré |
| 52 | négatif | positif | « petit film sympathique… en beaucoup moins bien » | **discutable** : avis mitigé |
| 135 | négatif | positif | « agréable, sans prétention, plutôt bien joué, un bon moment » | **l'étiquette** : le texte est clairement positif, la note de l'auteur ne lui correspond sans doute pas |

**Conclusion** : une partie des « erreurs » n'en sont pas vraiment. Les étiquettes Allociné viennent des notes des spectateurs, et certaines ne reflètent pas le texte (avis mitigés, note incohérente). Les vraies erreurs de Qwen sont des cas difficiles : **ironie**, **verdict à la fin** après une liste de défauts, **critiques hors sujet**. Ce bruit dans les étiquettes fixe un **plafond** : aucun modèle ne peut atteindre 100 %, ce qui explique en partie pourquoi CamemBERT et Qwen plafonnent tous les deux autour de 95–96 %.

## 4.6 Tes avis pièges

### À TOI (4.6)
Passe tes 3 avis pièges dans `classer` et complète le tableau. (Le LLM donne une réponse, pas une probabilité : écris simplement sa prédiction.)

In [61]:
# À TOI
a=classer("Après avoir réussi à mêler film horreur et teen-movie avec le génial Destination finale, le réalisateur James Wong s'est embarqué sous les sirènes d'Hollywood dans un film de science-fiction bourré d'action et d'effets spéciaux. Sur le papelard, The One avait l'air plutôt intéressant avec cette histoire de combat entre personnages de plusieurs dimensions parallèles. Et puis voir Jet Li dans un double rôle était une idée alléchante pour les fans de la star asiatique. En somme, The One promettait un bon film d'action science-fictionnel bourré d'affrontements câblés et d'effets visuels Matrixiens. Verdict : c'est sympa, c'est marrant, c'est bien foutu mais c'est hélas aussi très lourd. Tournant en rond au bout d'une vingtaine de minutes, le long-métrage s'avère très répétitif et surtout très peu captivant au final, le scénario n'arrivant jamais à aller plus loin que le bout de son nez. Quant au casting, c'est également peu convaincant... Jet Li peine à convaincre dans son double rôle de gentil paumé et de méchant diabolique tandis que son comparse de fortune Jason Statham (Ghosts of Mars) se régale à faire le big boy qui flingue tout sur son passage. S'ajoutent également au casting le polyvalent Delroy Lindo et la sexy Carla Gugino, campant respectivement le traqueur du bad guy et la comparse du héros. The One est donc une série B tout ce qu'il y a de plus banale où scènes d'action filmées au ralenti, effets visuels à outrance et musique rock omniprésente se fracassent constamment dans un gros bordel pas toujours digérable. Et si le film confirme une belle place dans le cinéma d'action pour Jason Statham, il nous fait amèrement regretter le passage Outre-Atlantique d'un Jet Li confiné aux rôles indigestes voire nanardesques.")
b=classer("Film pas mauvais, qui se laisse regarder. Très belle performance de Wahlberg pas habitué à jouer ce genre de rôle et qui réussit son pari.")
c=classer("Un film d'un grand intérêt, pas mauvais du tout.")
print(a["prediction"])
print(b["prediction"])
print(c["prediction"])


0
1
1


| Avis | Vraie réponse | TF-IDF + bigrammes | CamemBERT | Qwen 2.5 3B |
|---|---|---|---|---|
| Piège 1 (« génial ») | négatif | 22 % → négatif ✅ | 0,6 % → négatif ✅ | négatif ✅ |
| Piège 2 (« pas mauvais ») | positif | 38 % → négatif ❌ | 97,8 % → positif ✅ | positif ✅ |
| Piège 3 (inventé) | positif | 0,5 % → négatif ❌ | 97,8 % → positif ✅ | positif ✅ |

**Ma conclusion (4.6)** : comme CamemBERT, Qwen réussit les 3 pièges que TF-IDF ratait en partie. Un LLM lit l'avis en entier avec l'attention : il comprend la négation (« pas mauvais ») et voit que « génial » parle d'un autre film.

## 4.7 (Optionnel) Claude via l'API d'Anthropic

À faire **quand tu auras une clé API** (console.anthropic.com → Billing → crédits → API Keys). Un abonnement Claude.ai ne suffit pas : l'API est facturée à part.

### Préparer la clé
1. Dans le dossier `avis-allocine`, crée un fichier nommé exactement **`.env`** contenant une seule ligne : `ANTHROPIC_API_KEY=ta-clé-ici`
2. **Ne colle jamais ta clé dans le notebook** : le notebook ira sur GitHub, et une clé publiée peut être utilisée par n'importe qui à tes frais. Le `.env`, lui, est dans le `.gitignore`.

### Ce qui change par rapport à Ollama
| | Ollama | Claude |
|---|---|---|
| Format imposé | `format=SCHEMA` | `output_config={"format": {"type": "json_schema", "schema": SCHEMA}}` |
| Réponse | `reponse.message.content` | une **liste de blocs** `reponse.content` : un bloc `thinking` (sa réflexion) puis un bloc `text` (le JSON) |
| Tokens | `prompt_eval_count`, `eval_count` | `reponse.usage.input_tokens`, `reponse.usage.output_tokens` |
| Coût | 0 | **4 $ / million** de tokens d'entrée, **20 $ / million** en sortie (réflexion comprise) |

`effort: "low"` demande à Claude de peu réfléchir (la tâche est simple : moins cher, plus rapide). `fallbacks="default"` : si Claude refuse un avis (rare, ses filtres de sécurité peuvent se tromper), un autre modèle prend le relais.

In [ ]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("ANTHROPIC_API_KEY"):
    print("⚠️ Pas de clé : crée le fichier .env, puis redémarre le noyau")

client = anthropic.Anthropic()

MODELE_CLAUDE = "claude-opus-5-5"
PRIX_ENTREE = 4 / 1_000_000   # dollars par token d'entrée
PRIX_SORTIE = 20 / 1_000_000  # dollars par token de sortie


def appeler_claude(texte):
    """Envoie un avis à Claude et renvoie la réponse complète de l'API."""
    return client.beta.messages.create(
        model=MODELE_CLAUDE,
        max_tokens=2000,
        system=CONSIGNE,
        messages=[{"role": "user", "content": texte}],
        output_config={"effort": "low", "format": {"type": "json_schema", "schema": SCHEMA}},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    )


if os.getenv("ANTHROPIC_API_KEY"):
    reponse_c = appeler_claude(echantillon.loc[0, "review"])
    print("stop_reason :", reponse_c.stop_reason)
    for bloc in reponse_c.content:
        print(bloc.type, "→", getattr(bloc, "text", ""))
    print(reponse_c.usage.input_tokens, "tokens d'entrée,", reponse_c.usage.output_tokens, "tokens de sortie")

### À TOI (4.7)
1. Écris `classer_claude(texte)` sur le modèle de `classer`, avec un champ en plus : `"dollars"`. Récupère le texte du bloc dont le `type` vaut `"text"`. Si `reponse.stop_reason == "refusal"`, mets `prediction` à `None`.
2. **Pilote sur 20 avis** : accuracy, latence, coût total. **Estime le coût pour 500 et pour 20 000 avis** avant d'aller plus loin.
3. Si le coût te convient : les 500 avis, sauvegardés dans `data/claude_500.csv`, puis accuracy et marge d'erreur.
4. Ajoute une colonne Claude au tableau des avis pièges du 4.6.

In [ ]:
# À TOI


**Mes résultats (4.7)** : *à faire quand j'aurai une clé API.*

## 4.8 Le bilan

C'est **le tableau central du projet**. Latences mesurées à l’étape 5 avec mesurer_vitesse.py ; la colonne Claude sera remplie quand j'aurai une clé API.

| | TF-IDF + bigrammes | CamemBERT | Qwen 2.5 3B (local) | Claude (API) |
|---|---|---|---|---|
| Accuracy | 93,8 % (20 000 avis) | 95,8 % (20 000 avis) | **95,4 %** (500 avis, ± 1,8) | à mesurer |
| Avis d'entraînement | 160 000 | 20 000 | **0** | 0 |
| Entraînement | quelques minutes, CPU | ~4 min, GPU | aucun | aucun |
| Latence par avis (CPU) | 0,6 ms | 83 ms | ~0,27 s | à mesurer |
| Coût pour 1 000 avis | ~0 $ (mon PC) | ~0 $ (mon PC) | ~0 $ (mon PC) | à mesurer |
| Avis pièges réussis | 1 / 3 | 3 / 3 | 3 / 3 | à mesurer |
| Données envoyées à l'extérieur | non | non | non | oui (API) |

**Ma conclusion (4.8)** :
- **Le résultat surprenant** : un petit LLM local de 3 milliards de paramètres, **sans voir un seul avis étiqueté**, fait aussi bien que CamemBERT fine-tuné (la différence est dans la marge d'erreur) et mieux que TF-IDF entraîné sur 160 000 avis.
- **Pour un million d'avis par jour**, la vitesse et le coût comptent plus que tout. À ~0,27 s par avis, Qwen mettrait environ 75 heures sur mon PC : beaucoup trop pour une seule machine. Je choisirais **CamemBERT** (bien plus rapide quand on traite les avis par paquets sur GPU), ou **TF-IDF** si le budget est minimal et que 2 points d'accuracy en moins sont acceptables.
- **Pour un prototype à livrer demain, sans données étiquetées**, je choisirais un **LLM en zero-shot** : aucun entraînement, juste un prompt. Qwen en local si les données sont confidentielles ou le budget nul, une API comme Claude si on veut la meilleure qualité possible.
- **Les deux se combinent** : on peut utiliser un LLM pour **étiqueter** des milliers d'avis, puis entraîner un petit modèle rapide (TF-IDF ou CamemBERT) sur ces étiquettes pour la production. C'est une forme de **distillation**.